In [29]:
# Imports and file paths
# No CSV export needed — data goes straight from binary .nfr to DataFrame to plots
import pandas as pd
import plotly.express as px
import plotly.io as pio
from compile import compile_csv
from decode import decode_frame
from main import read_frames_from_file

# Use the VS Code renderer to avoid nbformat mime rendering issues
pio.renderers.default = "vscode"

DBC_PATH = "../NFR26CANDBC.csv"
DATA_PATH = "testData/26-05-10/LOG_0026.NFR"

## 1. Compile DBC and decode frames

`compile_csv()` builds a decode table from the DBC CSV (signal definitions: bit positions, scales, offsets).
`read_frames_from_file()` reads raw 18-byte frames from the binary `.nfr` file.
`decode_frame()` extracts signal values from each frame's bytes using bit masking and scale/offset math.

Each decoded signal becomes a row in an in-memory DataFrame — no intermediate CSV file is written.

# Signal Graphing

This notebook reads CAN signals directly from binary `.nfr` log files and graphs them interactively using Plotly. No intermediate CSV is needed.

**Data flow:** DBC CSV (signal definitions) + `.nfr` binary (raw CAN frames) → decode in memory → pandas DataFrame → interactive plots

In [30]:
# Step 1: compile_csv() parses DBC CSV into a decode table (frame_id -> message/signal definitions)
# Step 2: read_frames_from_file() reads 18-byte LogFrames from binary .nfr (timestamp, id, dlc, data)
# Step 3: decode_frame() bit-masks each signal from the raw bytes, applies scale/offset
# Step 4: each signal value becomes a row in the DataFrame — no CSV written to disk
decode_table = compile_csv(DBC_PATH)

signal_units = {}
for msg in decode_table.values():
    for sig in msg.signals:
        signal_units[(msg.frame_id, sig.name)] = sig.unit or ""

rows = []
for timestamp, frame_id, data in read_frames_from_file(DATA_PATH):
    decoded = decode_frame(frame_id, data, decode_table)
    if not decoded:
        continue
    msg = decode_table[frame_id]
    for signal_name, value in decoded.items():
        unit = signal_units.get((frame_id, signal_name), "")
        rows.append({
            "signal_name": signal_name,
            "timestamp_ms": timestamp,
            "value": value,
            "message_id": f"0x{frame_id:X}",
            "message_name": msg.name,
            "unit": unit,
            "sender": msg.sender,
        })

df = pd.DataFrame(rows)

# Check if the file was empty or only contained a header with no frames
if df.empty:
    print(f"No signals decoded from {DATA_PATH} — file may be empty or contain only a header.")
else:
    print(f"{len(df)} signal readings from {df['signal_name'].nunique()} unique signals")
    df.head(10)

11037 signal readings from 336 unique signals


## 2. Plot helper

In [31]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_signals_by_sender(df, max_subplots_per_fig=10):
    if df.empty:
        print("No signals decoded.")
        return
        
    for sender, sender_group in df.groupby('sender'):
        # Group by message and signal within this sender
        groups = list(sender_group.groupby(["message_name", "signal_name"]))
        
        # Chunk within the sender to avoid WebGL texture limits
        for chunk_idx in range(0, len(groups), max_subplots_per_fig):
            chunk = groups[chunk_idx : chunk_idx + max_subplots_per_fig]
            num_plots = len(chunk)
            
            fig = make_subplots(rows=num_plots, cols=1, subplot_titles=[
                f"{sig_name} — {msg_name} ({group['message_id'].iloc[0]})" 
                for (msg_name, sig_name), group in chunk
            ], vertical_spacing=0.05)

            for i, ((msg_name, sig_name), group) in enumerate(chunk):
                unit = group["unit"].iloc[0]
                y_label = f"{sig_name} ({unit})" if unit else sig_name
                
                fig.add_trace(
                    go.Scattergl(
                        x=group["timestamp_ms"],
                        y=group["value"],
                        mode="lines",
                        name=sig_name,
                    ),
                    row=i+1, col=1
                )
                
                fig.update_yaxes(title_text=y_label, row=i+1, col=1)
                if i == num_plots - 1:
                    fig.update_xaxes(title_text="Time (ms)", row=i+1, col=1)
            
            # Title like "IMU Signals (1 to 8)"
            start_idx = chunk_idx + 1
            end_idx = chunk_idx + num_plots
            if len(groups) > max_subplots_per_fig:
                title_text = f"{sender} Signals ({start_idx} to {end_idx})"
            else:
                title_text = f"{sender} Signals"
                
            fig.update_layout(
                height=300 * num_plots, 
                showlegend=False,
                title_text=title_text
            )
            
            fig.show()

## 3. Graph all signals

Loops over every unique (message, signal) pair in the DataFrame and creates an interactive plot for each. Skips if data is empty.

In [32]:
# Plot signals grouped by their sender (e.g. IMU, BMS, VCU) 
# and chunked to avoid WebGL limits
plot_signals_by_sender(df, max_subplots_per_fig=8)